# 02 Data Preprocessing

Preprocessing CRLMSSold*.csv data according to findings from the EDA (01_exploration.ipynb) to create clean CSV and prep for week 4.

## Set Up

Loading the same .csv files used in 01_exploration.ipynb

In [1]:
import pandas as pd
import numpy as np
import glob

pd.set_option('display.max_columns', 100)

Data_dir = '../../data' 

data = sorted(glob.glob(f'{Data_dir}/CRMLSSold*.csv'))

#error check
assert len(data) > 0, "No CRMLSSold*.csv files found."
print(f'Found {len(data)} file(s):')
for f in data:
    print(f'  {f}')

Found 6 file(s):
  ../../data\CRMLSSold202512.csv
  ../../data\CRMLSSold202601.csv
  ../../data\CRMLSSold202602.csv
  ../../data\CRMLSSold202603.csv
  ../../data\CRMLSSold202604.csv
  ../../data\CRMLSSold202605.csv


In [2]:
all_csv = []
raw_total = 0

# Apply filters to all files
for file in data:
    #print(f"Processing file: {file}")
    csv = pd.read_csv(file, low_memory=False)
    raw_total += len(csv)

    # restrict analysis per task doc
    csv = csv[(csv['PropertyType'] == 'Residential') & (csv['PropertySubType'] == 'SingleFamilyResidence')].copy()
    
    all_csv.append(csv)

df = pd.concat(all_csv, ignore_index=True)

#geo check: drop out-of-scope rows (non-California)
OUT_OF_SCOPE = ['Foreign Country', 'Other State']
n_before = len(df)
df = df[~df['CountyOrParish'].isin(OUT_OF_SCOPE)].reset_index(drop=True)
print(f'Dropped {n_before - len(df)} out-of-scope rows (non-California): {OUT_OF_SCOPE}')

# Reverify total files read
print(f'Total files read: {len(all_csv)}')
# Rows before filtering
print(f'Total rows read pre-filter: {raw_total:,}')
print(f'Total rows read post-filter (restrictions): {len(df):,}')

Dropped 3 out-of-scope rows (non-California): ['Foreign Country', 'Other State']
Total files read: 6
Total rows read pre-filter: 124,404
Total rows read post-filter (restrictions): 61,724


### Convert CloseDate to datetime

Necessary step since deduplication and week 4 task requires validity and consistent real dates.

In [3]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')

bad_dates = df['CloseDate'].isna().sum()
print(f'CloseDate parse failures: {bad_dates}')

# drop any date parse fails
if bad_dates > 0:
    df = df.dropna(subset=['CloseDate']).reset_index(drop=True)
    print(f'Dropped {bad_dates} rows with unparseable CloseDate.')

CloseDate parse failures: 0


## Deduplication

- EDA saw 33 duplicated ListingKeys
- Of the 33 duplicates, most pairs share the same ClosePrice, but conflict through CloseDates.
- Some pairs conflict on the ClosePrice

In [4]:
#quantify and inspect the conflict groups before touching anything

pre_dedup = len(df)
# print(f'amount of rows before deduplication: {pre_dedup}')

dup_rows = df[df['ListingKey'].duplicated(keep=False)]
price_nunique = dup_rows.groupby('ListingKey')['ClosePrice'].nunique()

conflict_keys = price_nunique[price_nunique > 1].index
print(f'Total Dupe ListingKeys: {len(price_nunique)}')
print(f'- Agreeing with ClosePrice: {(price_nunique == 1).sum()}')
print(f'- Conflict with ClosePrice: {len(conflict_keys)}')

Total Dupe ListingKeys: 33
- Agreeing with ClosePrice: 24
- Conflict with ClosePrice: 9


In [5]:
# Inspecting the conflicting groups
insp_cols = [col for col in ['ListingKey', 'ClosePrice', 'CloseDate', 'ListPrice']
                if col in df.columns]

display(df[df['ListingKey'].isin(conflict_keys)] .sort_values(['ListingKey', 'CloseDate'])[insp_cols])

,ListingKey,ClosePrice,CloseDate,ListPrice
17364,1118859284,875000.0,2026-01-16,870000.0
26098,1118859284,874400.0,2026-02-06,870000.0
6184,1137564166,425000.0,2025-12-26,419900.0
15843,1137564166,435000.0,2026-01-16,419900.0
14862,1144635053,1578331.0,2026-01-29,1659248.0
24413,1144635053,1714623.0,2026-02-12,1659248.0
48680,1145660298,1200000.0,2026-04-03,1250000.0
61148,1145660298,1000000.0,2026-05-04,1250000.0
11362,1149466652,1230000.0,2026-01-20,965000.0
34792,1149466652,1205900.0,2026-03-19,965000.0


### Dealing with Duplicates (ListingKey)

To deal with the duplicates that are re-reported in later month, we will use the recent version of the MLS as sorted using CloseDate (and ContractStatusChangeDate for verification).

In [6]:
sort_cols = ['ListingKey', 'CloseDate']
if 'ContractStatusChangeDate' in df.columns:
    sort_cols.append('ContractStatusChangeDate')

df = (df.sort_values(sort_cols)
        .drop_duplicates(subset='ListingKey', keep='last')
        .reset_index(drop=True))

print(f'Rows removed by dedup: {pre_dedup - len(df)}')
print(f'Remaining rows: {len(df):,}')
assert df['ListingKey'].is_unique

Rows removed by dedup: 33
Remaining rows: 61,691


### Implausible Values

Values that were flaged from the EDA

| Field | Problem | Count (pre-dedup) |
|---|---|---|
| ClosePrice | min = $1.75 reall small value | 4 |
| ClosePrice | max $796M extreme outlier | 7 |
| LivingArea | 0 size is uninhabitable| 19 |
| BedroomsTotal | value of 0, could be a studio needs clarifying | 29 |
| BathroomsTotalInteger | value of 0, not normal for residences| 24 |
| LotSizeAcres | 0 values, potential unit errors (43,560 = 1 acre in sq ft; max 60,113 acres) | 185 where LotSizeAcres = 0 |

In [7]:
# reverify EDA flags
checks = {
    'LivingArea == 0':            (df['LivingArea'] == 0).sum(),
    'BedroomsTotal == 0':         (df['BedroomsTotal'] == 0).sum(),
    'BathroomsTotalInteger == 0': (df['BathroomsTotalInteger'] == 0).sum(),
    'LotSizeAcres == 0':          (df['LotSizeAcres'] == 0).sum(),
    'ClosePrice < $10,000':       (df['ClosePrice'] < 10_000).sum(),
    'ClosePrice > $100M':         (df['ClosePrice'] > 100_000_000).sum(),
}
for label, n in checks.items():
    print(f'{label:<30} {n:>6,} rows')

LivingArea == 0                    19 rows
BedroomsTotal == 0                 29 rows
BathroomsTotalInteger == 0         24 rows
LotSizeAcres == 0                 185 rows
ClosePrice < $10,000                4 rows
ClosePrice > $100M                  7 rows


 ### i. Zero-value structure fields

 From above, it's difficult to accept that a single-family residence is missing a bedroom, bathroom, and is uninhabitable (LivingArea). These are more likely to be data entry issues rather than true values.

Decicion: Drop zero-value entries of BedroomsTotal, BathroomIntegersTotal, and LivingArea.

In [8]:
n_before = len(df)
print(n_before)

implausible_structure = (
    (df['LivingArea'] == 0) |
    (df['BedroomsTotal'] == 0) |
    (df['BathroomsTotalInteger'] == 0)
)
df = df[~implausible_structure].reset_index(drop=True)

print(f'Rows dropped for zero-value fields: {n_before - len(df):,}')
print(f'Remaining rows: {len(df):,}')

61691
Rows dropped for zero-value fields: 54
Remaining rows: 61,637


### ii. ClosePrice: Establishing boundaries

Some of the ClosePrice outliers can only be removed due to the impact on the rest of the distribution. The following code investigates the flagged entries each potential issue causing the flags:
- unit calculation errors
- abiguity

In [9]:
# Inspect the outlier rows
MIN = 10_000
MAX = 100_000_000

#Plausibilty/further cleaning
PPSF_MAX = 15_000


out_of_bounds = ~df['ClosePrice'].between(MIN, MAX)
implausible_ppsf = (df['ClosePrice'] / df['LivingArea']) > PPSF_MAX #revision
flagged = out_of_bounds | implausible_ppsf #revision

inspect_cols = [col for col in ['ListingKey', 'ClosePrice', 'ListPrice',
                                'LivingArea', 'CountyOrParish', 'City', 'CloseDate']
                if col in df.columns]
inspect = df.loc[flagged, inspect_cols].copy()
inspect['Close/List'] = (inspect['ClosePrice'] / inspect['ListPrice']).round(4)
inspect['$/sqft'] = (inspect['ClosePrice'] / inspect['LivingArea']).round(0) #revision

print(f'Rows outside [${MIN:,}, ${MAX:,}]: {out_of_bounds.sum()}')
display(inspect.sort_values('ClosePrice'))

Rows outside [$10,000, $100,000,000]: 11


,ListingKey,ClosePrice,ListPrice,LivingArea,CountyOrParish,City,CloseDate,Close/List,$/sqft
13831,1144223437,1.750000e+00,1749000.0,3513.0,Riverside,Palm Desert,2026-01-30,0.0000,0.0
34754,1151459976,6.850000e+02,913655.0,2980.0,Contra Costa,Oakley,2026-04-29,0.0007,0.0
24421,1147508351,8.000000e+03,8000.0,984.0,San Bernardino,Trona,2026-02-04,1.0000,8.0
682,1110035161,8.300000e+03,2000000.0,3886.0,Los Angeles,Altadena,2026-01-17,0.0042,2.0
49595,1154925192,4.872000e+07,5150000.0,1800.0,San Diego,Encinitas,2026-05-15,9.4602,27067.0
38045,1152020272,9.797250e+07,975000.0,2254.0,San Diego,Oceanside,2026-05-22,100.4846,43466.0
8099,1135757473,4.720000e+08,479000.0,1910.0,San Bernardino,Victorville,2026-01-19,985.3862,247120.0
23153,1147256230,6.150000e+08,625000.0,2065.0,Riverside,Lake Elsinore,2026-02-27,984.0000,297821.0
44178,1153305774,6.450000e+08,645000.0,1104.0,San Diego,Oceanside,2026-04-22,1000.0000,584239.0
25462,1148201765,6.642500e+08,750000.0,1992.0,San Diego,Chula Vista,2026-03-26,885.6667,333459.0


In [10]:
# Potential unit corrections

FACTORS = [1, 1e-3, 1e-2, 1e-1, 1e3, 1e6]

# 1 = keep, 
# 1e-3 = divide by 1,000 

#revision: added more factors to check for potential unit errors
# 1e-2 = divide by 100
# 1e-1 = divide by 10

# 1e3 = multiply by 1,000
# 1e6 = multiply by 1M

RATIO = (0.80, 1.25) 

out= df.index[flagged]
df['ClosePrice_repaired'] = 0

repaired, kept, dropped = [], [], []

for i in out:
    price = df.at[i, 'ClosePrice']
    lst   = df.at[i, 'ListPrice'] #compare to the listing price to see if the price is a unit error
    if pd.isna(lst) or lst <= 0: #doesn't have a valid comparing price so drop
        dropped.append((i, price, lst, []))
        continue

    matches = []
    for f in FACTORS:
        corrected = price * f
        ratio_ok  = RATIO[0] <= corrected / lst <= RATIO[1]
        bounds_ok = (f == 1) or (MIN <= corrected <= MAX)
        if ratio_ok and bounds_ok:
            matches.append(f)

    if matches == [1]:
        kept.append((i, price, lst)) #price is already in bounds and ratio is okay
    elif len(matches) == 1:
        df.at[i, 'ClosePrice'] = price * matches[0]
        df.at[i, 'ClosePrice_repaired'] = 1
        repaired.append((i, price, price * matches[0], matches[0]))
    else: # IS ABIGUOUS CASE
        dropped.append((i, price, lst, matches))
    
print(f'Out-of-bounds rows examined: {len(out)}\n')
print(f'# REPAIRED rows (via unit decoding): {len(repaired)}\n')

for i, old, new, factor in repaired:
    key = df.at[i, 'ListingKey']
    lst = df.at[i, 'ListPrice']
    closed = df.at[i, 'CloseDate'].date()
    ratio = new / lst
    status = "PASS" if RATIO[0] <= ratio <= RATIO[1] else "FAIL"

    print(f' ListingKey {key} | CloseDate: {closed}:\n'
          f'    ListPrice: ${lst:,.0f} \n'
          f'    Original ClosePrice: ${old:,.2f} \n'
          f'    --> Calculation: ${old:,.2f} × {factor:g} \n'
          f'    ==> Repaired ClosePrice: ${new:,.0f} \n' 
          f'    Ratio Check: {new:,.0f} / {lst:,.0f} = {ratio:.3f} ({status})\n')
    
print(f'# REMAINING rows as Original ClosePrice: {len(kept)}')

for i, price, lst in kept:
    key = df.at[i, 'ListingKey']
    closed = df.at[i, 'CloseDate'].date()
    ratio = price / lst
    status = "PASS" if RATIO[0] <= ratio <= RATIO[1] else "FAIL"

    print(f'ListingKey {key} | CloseDate: {closed}:\n'
          f'    ListPrice: ${lst:,.0f}\n'
          f'    Original ClosePrice (Kept): ${price:,.2f}\n'
          f'    Ratio Check: {price:,.0f} / {lst:,.0f} = {ratio:.3f} ({status})\n')


print(f'# DROPPED rows (ambiguous): {len(dropped)}')

for i, price, lst, matches in dropped:
    key = df.at[i, 'ListingKey']
    closed = df.at[i, 'CloseDate'].date()

    print(f'ListingKey {key} | CloseDate: {closed}:\n'
        f'    ListPrice: ${lst:,.0f}\n'
        f'    Original ClosePrice: ${price:,.2f}\n'
        f'    Matching Factors: {matches if matches else "None"}\n'
        f'    Decision: DROPPED (ambiguous or no valid correction)\n')

drop_i = [i for i, _, _, _ in dropped]
n_before = len(df)
df = df.drop(index=drop_i).reset_index(drop=True)
print(f'Rows dropped: {n_before - len(df)} | Total Rows Remaining in df: {len(df):,}')

Out-of-bounds rows examined: 13

# REPAIRED rows (via unit decoding): 10

 ListingKey 1135757473 | CloseDate: 2026-01-19:
    ListPrice: $479,000 
    Original ClosePrice: $472,000,000.00 
    --> Calculation: $472,000,000.00 × 0.001 
    ==> Repaired ClosePrice: $472,000 
    Ratio Check: 472,000 / 479,000 = 0.985 (PASS)

 ListingKey 1144223437 | CloseDate: 2026-01-30:
    ListPrice: $1,749,000 
    Original ClosePrice: $1.75 
    --> Calculation: $1.75 × 1e+06 
    ==> Repaired ClosePrice: $1,750,000 
    Ratio Check: 1,750,000 / 1,749,000 = 1.001 (PASS)

 ListingKey 1147256230 | CloseDate: 2026-02-27:
    ListPrice: $625,000 
    Original ClosePrice: $615,000,000.00 
    --> Calculation: $615,000,000.00 × 0.001 
    ==> Repaired ClosePrice: $615,000 
    Ratio Check: 615,000 / 625,000 = 0.984 (PASS)

 ListingKey 1148201765 | CloseDate: 2026-03-26:
    ListPrice: $750,000 
    Original ClosePrice: $664,250,000.00 
    --> Calculation: $664,250,000.00 × 0.001 
    ==> Repaired ClosePr

Notes:

- 8/11 rows repaired due to unit inconsistencies
- 1/11 rows maintained as-is due to reasonable connection to its ListPrice.
- 2/11 rows dropped entirely due to ambiguity

### iii. LotSizeAcres: 

The following strategy investigates the flagged entries each potential issue causing the flags:
- extreme skew
- log transform does not fix
- unit-conversion errors/consistency

Using LotSizeSquare to offer reference and potential solutions.

In [11]:
# Cross check units through LotSizeAcres vs LotSizeSquare calculations

if 'LotSizeSquareFeet' not in df.columns:
    print('LotSizeSquareFeet missing, skipping cross-check.')
else:
    acres = df['LotSizeAcres']
    sqft  = df['LotSizeSquareFeet']

    acres_ok = acres.notna() & (acres > 0)
    sqft_ok  = sqft.notna() & (sqft > 0)
    both     = acres_ok & sqft_ok

    consistent     = both & np.isclose(sqft, acres * 43_560, rtol=0.01)
    copy_error     = both & ~consistent & np.isclose(acres, sqft, rtol=0.001)
    mismatch = both & ~consistent & ~copy_error
    repair_src  = ~acres_ok & sqft_ok #acres unusable, sqft available
    both_unusable  = ~acres_ok & ~sqft_ok

    print(f'Rows both populated & nonzero:     {both.sum():>7,}')
    print(f'Consistent (ratio ~43,560):        {consistent.sum():>7,}')
    print(f'Copy error (acres == sqft):        {copy_error.sum():>7,}')
    print(f'Other mismatch:                    {mismatch.sum():>7,}')
    print(f'\nLotSizeAcres unusable, LotSizeSquareFeet available: {repair_src.sum():>7,}')
    print(f'Both variables unusable:           {both_unusable.sum():>7,}')

    # Flags as suspected sq-ft entries:
    suspected = acres > 640
    print(f'\nSuspected sq-ft entries (acres > 640):   {suspected.sum():,}')
    print(f'  - copy errors (sqft == acres):         {(suspected & copy_error).sum():,}')
    print(f'  = "consistent" :                       {(suspected & consistent).sum():,}')

    if mismatch.sum() > 0:
        print('\nSample of mismatches:')
        display(df.loc[mismatch,
                       ['ListingKey', 'LotSizeAcres', 'LotSizeSquareFeet']].head(10))

Rows both populated & nonzero:      60,366
Consistent (ratio ~43,560):         60,363
Copy error (acres == sqft):              0
Other mismatch:                          3

LotSizeAcres unusable, LotSizeSquareFeet available:     145
Both variables unusable:             1,122

Suspected sq-ft entries (acres > 640):   45
  - copy errors (sqft == acres):         0
  = "consistent" :                       43

Sample of mismatches:


,ListingKey,LotSizeAcres,LotSizeSquareFeet
3848,1120172649,0.0002,6.72
15221,1144778109,0.0003,11.76
43743,1153245042,0.0001,2.63


In [12]:
MIN_ACRES = 0.01   # ~436 sq ft
MAX_ACRES = 640    # 1 sq-mi
SQFT_PER_ACRE = 43_560

lot = df['LotSizeAcres'].copy()
og_na = lot.isna().sum()

#zero lot size -> NaN
zero = lot.index[lot == 0]
lot.loc[zero] = np.nan
print(f'Zero (0) lot size -> NaN: {len(zero)} rows')

#suspected sq-ft entries
suspect_idx = lot.index[lot > MAX_ACRES]
print(f'\nSuspected sq-ft entries (> {MAX_ACRES} acres): {len(suspect_idx)} rows\n')

repaired = 0
for i in suspect_idx:
    val  = lot.at[i]
    conv = val / SQFT_PER_ACRE
    ok   = MIN_ACRES <= conv <= MAX_ACRES
    key    = df.at[i, 'ListingKey']
    closed = df.at[i, 'CloseDate'].date()

    # Cross-reference sq-ft colum:
    # sqft == acres value
    # sqft == acres x 43,560

    xref = ''
    if 'LotSizeSquareFeet' in df.columns:
        sqft = df.at[i, 'LotSizeSquareFeet']
        if pd.isna(sqft):
            xref = '    Cross-ref LotSizeSquareFeet: missing\n'
        elif np.isclose(sqft, val, rtol=0.001):
            xref = (f'    Cross-ref LotSizeSquareFeet: {sqft:,.0f} '
                    f'== acres value (COPY ERROR confirmed)\n')
        elif np.isclose(sqft, val * SQFT_PER_ACRE, rtol=0.01):
            xref = (f'    Cross-ref LotSizeSquareFeet: {sqft:,.0f} '
                    f'= acres x 43,560 (auto-derived; sq-ft column equally corrupted)\n')
        else:
            xref = (f'    Cross-ref LotSizeSquareFeet: {sqft:,.0f} '
                    f'(inconsistent with both readings)\n')

    status   = 'PASS' if ok else 'FAIL'
    decision = f'REPAIRED -> {conv:.4f} acres' if ok else 'set to NaN (conversion implausible)'
    print(f'  ListingKey {key} | CloseDate: {closed}:\n'
          f'    Original LotSizeAcres: {val:,.4f}\n'
          + xref +
          f'    --> Calculation: {val:,.4f} / {SQFT_PER_ACRE:,}\n'
          f'    ==> Converted: {conv:.4f} acres\n'
          f'    Plausibility Check: {MIN_ACRES} <= {conv:.4f} <= {MAX_ACRES} ({status})\n'
          f'    Decision: {decision}\n')

    if ok:
        lot.at[i] = conv
        repaired += 1
    else:
        lot.at[i] = np.nan

# small nonzero values -> NaN
small = lot.index[lot < MIN_ACRES]
print(f'Small (> 0, < {MIN_ACRES} acres) -> NaN: {len(small)} rows')
SHOW = 10

for i in small[:SHOW]:
    key = df.at[i, 'ListingKey']
    print(f'  ListingKey {key}: {lot.at[i]:.6f} acres (~{lot.at[i]*SQFT_PER_ACRE:,.0f} sq ft) -> NaN')
if len(small) > SHOW:
    print(f'  ... and {len(small) - SHOW} more')
lot.loc[small] = np.nan

# recover from LotSizeSquareFeet where acres is unusable
recovered, rejected = 0, 0
if 'LotSizeSquareFeet' in df.columns:
    cand = lot.index[lot.isna()
                     & df['LotSizeSquareFeet'].notna()
                     & (df['LotSizeSquareFeet'] > 0)]
    print(f'\nAcres unusable, sqft populated: {len(cand)} candidate rows')
    shown = 0
    for i in cand:
        sqft = df.at[i, 'LotSizeSquareFeet']
        conv = sqft / SQFT_PER_ACRE
        ok   = MIN_ACRES <= conv <= MAX_ACRES
        if shown < SHOW:
            key = df.at[i, 'ListingKey']
            verdict = f'RECOVERED: {conv:.4f} acres' if ok else 'rejected (outside plausible range)'
            print(f'  ListingKey {key}: {sqft:,.0f} sq ft / {SQFT_PER_ACRE:,} = {conv:.4f} -> {verdict}')
            shown += 1
        if ok:
            lot.at[i] = conv
            recovered += 1
        else:
            rejected += 1
    if len(cand) > SHOW:
        print(f'and {len(cand) - SHOW} more candidates')
    print(f' Recovered: {recovered} | Rejected: {rejected}')

df['LotSizeAcres'] = lot
df['LotSizeAcres_imputed'] = lot.isna() #flag

Zero (0) lot size -> NaN: 185 rows

Suspected sq-ft entries (> 640 acres): 45 rows

  ListingKey 1107719378 | CloseDate: 2025-12-31:
    Original LotSizeAcres: 1,888.0000
    Cross-ref LotSizeSquareFeet: 82,241,280 = acres x 43,560 (auto-derived; sq-ft column equally corrupted)
    --> Calculation: 1,888.0000 / 43,560
    ==> Converted: 0.0433 acres
    Plausibility Check: 0.01 <= 0.0433 <= 640 (PASS)
    Decision: REPAIRED -> 0.0433 acres

  ListingKey 1116879563 | CloseDate: 2026-03-13:
    Original LotSizeAcres: 4,000.0000
    Cross-ref LotSizeSquareFeet: 174,240,000 = acres x 43,560 (auto-derived; sq-ft column equally corrupted)
    --> Calculation: 4,000.0000 / 43,560
    ==> Converted: 0.0918 acres
    Plausibility Check: 0.01 <= 0.0918 <= 640 (PASS)
    Decision: REPAIRED -> 0.0918 acres

  ListingKey 1119966560 | CloseDate: 2025-12-12:
    Original LotSizeAcres: 6,900.0000
    Cross-ref LotSizeSquareFeet: 300,564,000 = acres x 43,560 (auto-derived; sq-ft column equally corrupte

In [13]:
# Summary
print(f'\nSummary:')
print(f'  Originally missing:            {og_na:,}')
print(f'  Zeros -> NaN:                  {len(zero):,}')
print(f'  Sq-ft entries repaired:        {repaired:,}')
print(f'  Suspected but unconvertible:   {len(suspect_idx) - repaired:,}')
print(f'  Too-small -> NaN:              {len(small):,}')
print(f'  Recovered from sqft column:    {recovered:,}')
print(f'  Total NaN awaiting imputation: {lot.isna().sum():,} ({lot.isna().mean()*100:.2f}% of rows)')
print(f'  Post-repair max LotSizeAcres:  {lot.max():,.2f}')


Summary:
  Originally missing:            1,082
  Zeros -> NaN:                  185
  Sq-ft entries repaired:        45
  Suspected but unconvertible:   0
  Too-small -> NaN:              12
  Recovered from sqft column:    0
  Total NaN awaiting imputation: 1,279 (2.08% of rows)
  Post-repair max LotSizeAcres:  450.00


Notes: 
- The LotSizeSquareFeet variable served as a cross-check, not a repair source
    - it is auto-derived from acres, so the 45 unit-error entries were repaired via the ÷43,560 heuristic (all 45 successfully, validated by their conversions landing in the normal 0.05–1.38 acre range).
- 185 zero-valued entries and 12 implausibly small lot sizes were recoded as missing (NaN) since could not be reliably interpreted.
- No suspected square-foot entries remained unconvertible; all 45 identified unit inconsistencies were successfully repaired.
- attempted recovery of unusable lot sizes from LotSizeSquareFeet (157 candidates) recovered 0 every candidate value was near-zero junk rejected by the plausibility guard, confirming the column carries no independent information.
- The 1,279 remaining NaN entries are flagged and passed to imputation.

## Missing Values

Handling missing values as found in 01_exploration.ipynb
- decide whether to drop, impute, or flag

### i. DROP

Dropping columns that are 100% null dynamically for any dataset.
Re-performance of the missingness column of 01_exploration.ipynb.

In [14]:
cols = df.columns[df.isna().all()].tolist()

print(f'Dropping {len(cols)} columns that are 100% null:')
for col in cols:
    print(f'  {col}')

df = df.drop(columns=cols)
print(f'\nColumns remaining: {df.shape[1]}')

Dropping 8 columns that are 100% null:
  FireplacesTotal
  AboveGradeFinishedArea
  TaxAnnualAmount
  TaxYear
  ElementarySchoolDistrict
  BusinessType
  CoveredSpaces
  MiddleOrJuniorSchoolDistrict

Columns remaining: 72


Note:
- Dynamically identified and dropped 8 columns that contained 100% missing values
- The pipeline automatically adapts to future datasets with different fully-null columns.
- 72 columns remained after removing the fully-null features.

### ii. Boolean Fields

- standardizing NaN by setting as 'False' for bool columns


In [15]:
#check for bool cols
yn_cols = [col for col in df.columns if col.endswith('YN')]
print(f'{len(yn_cols)} YN columns found: {yn_cols}\n')

# standardizing the representations
truthy_map = {True: True, False: False, 'True': True, 'False': False,
              'Y': True, 'N': False, 1: True, 0: False, 1.0: True, 0.0: False}
for col in yn_cols:
    df[col] = df[col].map(truthy_map)
    df[col] = df[col].fillna(False).astype(bool)

MINORITY = 0.001  # minority class

drop_yn = []
for col in yn_cols:
    true_share = df[col].mean()
    minority = min(true_share, 1 - true_share)
    status = 'DROP (near-constant)' if minority < MINORITY else 'keep'
    print(f'  {col:<28} True: {true_share*100:6.2f}%   -> {status}')
    if minority < MINORITY:
        drop_yn.append(col)

df = df.drop(columns=drop_yn)
print(f'\nDropped {len(drop_yn)} near-constant YN columns | Columns remaining: {df.shape[1]}')

7 YN columns found: ['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

  ViewYN                       True:  57.03%   -> keep
  WaterfrontYN                 True:   0.06%   -> DROP (near-constant)
  BasementYN                   True:   2.36%   -> keep
  PoolPrivateYN                True:  15.58%   -> keep
  AttachedGarageYN             True:  73.46%   -> keep
  FireplaceYN                  True:  72.35%   -> keep
  NewConstructionYN            True:   3.48%   -> keep

Dropped 1 near-constant YN columns | Columns remaining: 71


Note:

- Standardized 7 YN indicator columns into consistent Boolean (True/False) values by mapping mixed representations (Y/N, True/False, 1/0) to a common format.
- Dropped 1 near-constant Boolean feature (WaterfrontYN) after identifying a minority class frequency below the 0.1% threshold.
- Retained the remaining 6 Boolean features, as each contained sufficient variation to provide potential predictive value.
- 71 columns remained after removing the near-constant feature.

### iii. Missing strong predictors

- LivingArea / BedroomsTotal / BathroomsTotalInteger
- checking for missingness again and dropping from the dataset

In [16]:
n_before = len(df)
df = df.dropna(subset=['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger']).reset_index(drop=True)
print(f'Rows dropped for missing key predictors: {n_before - len(df):,}')
print(f'Remaining rows: {len(df):,}')

Rows dropped for missing key predictors: 31
Remaining rows: 61,604


- These features are considered essential predictors for modeling housing prices, so incomplete records were removed rather than imputed.
- 61,607 rows remained after filtering for complete key predictor information.

### iv. LotSizeAcres: Impute test


In [17]:
# First day of the most recent month in the data = start of the (future) test period
test_month_start = df['CloseDate'].max().to_period('M').to_timestamp()
fit_mask = df['CloseDate'] < test_month_start

lot_median = df.loc[fit_mask, 'LotSizeAcres'].median()
n_filled = df['LotSizeAcres'].isna().sum()

df['LotSizeAcres'] = df['LotSizeAcres'].fillna(lot_median)

print(f'Test month starts: {test_month_start.date()} (rows on/after this date excluded from the median)')
print(f'Median LotSizeAcres (fit on pre-test-month rows): {lot_median:.4f} acres')
print(f'Rows imputed: {n_filled:,} (flagged in LotSizeAcres_imputed)')

Test month starts: 2026-05-01 (rows on/after this date excluded from the median)
Median LotSizeAcres (fit on pre-test-month rows): 0.1671 acres
Rows imputed: 1,278 (flagged in LotSizeAcres_imputed)


Note:
- Imputed 1,278 missing LotSizeAcres values using the median (0.1671 acres) calculated from pre-test-month observations
- Excluded records from the test month (2026-05-01 start) when computing the median to prevent data leakage into the training process.
- All imputed values were flagged in the LotSizeAcres_imputed indicator column to preserve transparency for downstream analysis.

### v. Missingness

This audit lists what still contains NaNs, seeding directly into the keep/drop
decisions of the next section.

In [18]:
remaining = (df.isna().mean() * 100).round(2)
remaining = remaining[remaining > 0].sort_values(ascending=False)

if len(remaining) == 0:
    print('No missing values.')
else:
    print(f'{len(remaining)} columns still contain NaNs (% missing):')
    print(remaining.to_string())

40 columns still contain NaNs (% missing):
BelowGradeFinishedArea     99.28
BuilderName                95.49
LotSizeDimensions          93.85
BuildingAreaTotal          93.37
CoBuyerAgentFirstName      90.60
MiddleOrJuniorSchool       87.26
ElementarySchool           87.24
HighSchool                 83.14
CoListAgentFirstName       76.82
CoListOfficeName           76.74
CoListAgentLastName        76.74
AssociationFeeFrequency    73.85
SubdivisionName            64.55
MainLevelBedrooms          38.95
Flooring                   36.41
AssociationFee             28.48
HighSchoolDistrict         26.88
MLSAreaMajor               14.14
Stories                    10.52
BuyerOfficeAOR             10.47
Levels                      7.49
BuyerAgentAOR               5.92
GarageSpaces                3.77
LotSizeArea                 1.75
LotSizeSquareFeet           1.75
BuyerOfficeName             1.17
ListAgentFirstName          0.60
BuyerAgentFirstName         0.42
OriginalListPrice           0.22


Note:

- 40 columns still contained some missing values, with the highest missingness concentrated in optional/descriptive fields such as BelowGradeFinishedArea, BuilderName, LotSizeDimensions, and school-related columns.
- This audit is used to guide the next keep/drop decisions rather than immediately removing all columns with missing values.

## 4. Feature Selection & Leakage

- Dealing with data leakage prevention
- unique values
- Non or weak correlations with the target variable


### i. Preventing Data Leakage

In [19]:
price_like = [col for col in df.columns
              if 'Price' in col and col not in ('ClosePrice', 'ClosePrice_repaired')]
print(f'Dropping {len(price_like)} price-like columns as leakage:')
for col in price_like:
    print(f'  {col}')
df = df.drop(columns=price_like)

Dropping 2 price-like columns as leakage:
  OriginalListPrice
  ListPrice


### ii. Explicit keep


In [20]:
id_cols      = ['ListingKey', 'CloseDate']
target_cols  = ['ClosePrice']
feature_cols = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
                'LotSizeAcres', 'LotSizeAcres_imputed']
yn_cols      = [col for col in df.columns if col.endswith('YN')]  #sourced from 3ii.
cat_cols     = ['CountyOrParish']
ref_cols     = ['PostalCode', #raw ref
                'ClosePrice_repaired']

keep = id_cols + target_cols + feature_cols + yn_cols + cat_cols + ref_cols
not_found = [col for col in keep if col not in df.columns]
assert not not_found, f'Expected columns missing from this extract: {not_found}'

dropped = [col for col in df.columns if col not in keep]
print(f'Dropping remaining non-feature columns: {len(dropped)}')

df = df[keep].copy()
print(f'Columns kept ({df.shape[1]}):')
for col in df.columns:
    print(f'  {col}')

Dropping remaining non-feature columns: 52
Columns kept (17):
  ListingKey
  CloseDate
  ClosePrice
  LivingArea
  BedroomsTotal
  BathroomsTotalInteger
  LotSizeAcres
  LotSizeAcres_imputed
  ViewYN
  BasementYN
  PoolPrivateYN
  AttachedGarageYN
  FireplaceYN
  NewConstructionYN
  CountyOrParish
  PostalCode
  ClosePrice_repaired


### iii. PostalCode: ensuring consistency

In [21]:
df['PostalCode'] = (df['PostalCode'].astype('string')
                    .str.replace(r'\.0$', '', regex=True))
print(df['PostalCode'].head())

0    92103
1    92647
2    96094
3    90003
4    90740
Name: PostalCode, dtype: string


Notes:

- i. Preventing Data Leakage
    - Dropped 2 price-related variables (OriginalListPrice and ListPrice) to prevent data leakage during model training.
    - Retained ClosePrice as the target variable and ClosePrice_repaired as an audit flag indicating whether the target value was unit-corrected.
- ii. Explicit Keep
    - Explicitly retained 17 variables required for modeling, including the target, selected predictors, Boolean indicators, categorical features, and audit/reference columns.
    - Dropped the remaining 52 non-feature columns to create a streamlined modeling dataset.
    - PostalCode and ClosePrice_repaired were retained for future feature engineering and auditing but are not included as predictive features in the current baseline model.
- iii. PostalCode
    - Standardized and converted PostalCode to str data type to preserve leading zeros and ensure consistent formatting, also removing trailing decimal notations


## Encoding

Converting the categorical fields to numeric

- Boolean Fields
- CountyOrParish

In [22]:
# Boolean Fields
bool_cols = [col for col in df.columns if col.endswith('YN')] + ['LotSizeAcres_imputed']
df[bool_cols] = df[bool_cols].astype(int)
print("Converted to 0/1 int:")
for col in bool_cols:
    print(f"  {col}")

Converted to 0/1 int:
  ViewYN
  BasementYN
  PoolPrivateYN
  AttachedGarageYN
  FireplaceYN
  NewConstructionYN
  LotSizeAcres_imputed


In [23]:
# CountyOrParish

null_county = df['CountyOrParish'].isna().sum()
if null_county:
    print(f'Filling {null_county} missing CountyOrParish values with "Unknown"')
    df['CountyOrParish'] = df['CountyOrParish'].fillna('Unknown')

county_dummies = pd.get_dummies(df['CountyOrParish'], prefix='County', dtype=int)
county_dummies.columns = [col.replace(' ', '_') for col in county_dummies.columns]
df = pd.concat([df, county_dummies], axis=1)

print(f'Encoded CountyOrParish dummy columns: {county_dummies.shape[1]}')

Encoded CountyOrParish dummy columns: 56


Notes:

- i. Boolean Fields
    - Converted 7 Boolean variables (YN fields and LotSizeAcres_imputed) from Boolean values to 0/1 integer format.
- ii. CountyOrParish
    - One-hot encoded CountyOrParish into 58 binary indicator columns for use in modeling.
    - Standardized dummy column names by replacing spaces with underscores.

## Normalizing

ClosePrice and LivingArea
LotSizeAcres
BedroomsTotal / BathroomsTotalInterval


In [24]:
results = []

for col in ['ClosePrice', 'LivingArea', 'LotSizeAcres']:
    raw_skew = df[col].skew()
    log1p_skew = np.log1p(df[col]).skew()
    log_skew   = np.log(df[col]).skew()
    print(f'{col:<15} raw: {raw_skew:8.2f} | log1p: {log1p_skew:8.2f} | log: {log_skew:8.2f}')

df['ClosePrice_log']   = np.log1p(df['ClosePrice'])
df['LivingArea_log']   = np.log1p(df['LivingArea'])
df['LotSizeAcres_log'] = np.log(df['LotSizeAcres'])

print("\nAdded normalized features:")
print("  - ClosePrice_log")
print("  - LivingArea_log")
print("  - LotSizeAcres_log")

ClosePrice      raw:     8.62 | log1p:     0.50 | log:     0.50
LivingArea      raw:     2.95 | log1p:     0.32 | log:     0.32
LotSizeAcres    raw:    45.78 | log1p:     5.69 | log:     2.26

Added normalized features:
  - ClosePrice_log
  - LivingArea_log
  - LotSizeAcres_log


Notes:
- Applied a log transformation to reduce right-skewed distributions
- Creatied the features ClosePrice_log, LivingArea_log, and LotSizeAcres_log.
- The transformation substantially reduced skew for:
    - ClosePrice (12.69 → 0.51), 
    - LivingArea (2.95 → 0.32)
- LotSizeAcres remained positively skewed (45.78 → 2.26), indicating extreme outliers remain

## Train/Test Split
- Using the most recent month of available data as the test set, and the X months immediately preceding it as the training set. (2026-05)
- X is not fixed, treat the training window length as a tunable choice and experiment to determine the optimal value of X.

In [25]:
TRAIN_WINDOW_MONTHS = 3   # <-- X: the tunable training-window length

df['CloseMonth'] = df['CloseDate'].dt.to_period('M')
months = sorted(df['CloseMonth'].unique())
print(f'Months present: {[str(m) for m in months]}')

test_month = months[-1]
print(f'Test month (fixed): {test_month}')


def chrono_split(frame, x, test_month=test_month):
    """Most recent month = test set; the x months immediately preceding = train set."""
    train_months = [test_month - i for i in range(x, 0, -1)]
    train = frame[frame['CloseMonth'].isin(train_months)].copy()
    test = frame[frame['CloseMonth'] == test_month].copy()
    return train, test

Months present: ['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']
Test month (fixed): 2026-05


In [26]:
# Demonstrate the trade-off across every possible X
for x in range(1, len(months)):
    tr, te = chrono_split(df, x)
    print(f'X={x}: train {tr["CloseMonth"].min()} .. {tr["CloseMonth"].max()} '
          f'({len(tr):,} rows)  |  test {test_month} ({len(te):,} rows)')

X=1: train 2026-04 .. 2026-04 (12,011 rows)  |  test 2026-05 (12,007 rows)
X=2: train 2026-03 .. 2026-04 (23,167 rows)  |  test 2026-05 (12,007 rows)
X=3: train 2026-02 .. 2026-04 (31,697 rows)  |  test 2026-05 (12,007 rows)
X=4: train 2026-01 .. 2026-04 (39,162 rows)  |  test 2026-05 (12,007 rows)
X=5: train 2025-12 .. 2026-04 (49,597 rows)  |  test 2026-05 (12,007 rows)


In [27]:
train_df, test_df = chrono_split(df, TRAIN_WINDOW_MONTHS)
print(f'Chosen X = {TRAIN_WINDOW_MONTHS}')
print(f'Train: {train_df.shape[0]:,} rows   Test: {test_df.shape[0]:,} rows')

# Chronological sanity check: no train row may close on/after the test month
assert train_df['CloseMonth'].max() < test_month
assert (test_df['CloseMonth'] == test_month).all()

Chosen X = 3
Train: 31,697 rows   Test: 12,007 rows


Note:
- Used the most recent month (2026-05) as the fixed test set and evaluated training windows ranging from 1 to 5 months immediately preceding the test period.
- Selected a 3-month training window (2026-02 through 2026-04), resulting in 31,697 training rows and 12,007 test rows.
- Checked integrity by ensuring training observations occurred before the test month

## Export Cleaned CSV

- Not visible in the public Github repo for privacy. 
- Exported to local machine.

In [28]:
start_month = df['CloseDate'].min().strftime('%Y%m')
end_month   = df['CloseDate'].max().strftime('%Y%m')

OUTPUT_PATH = f"{Data_dir}/cleaned_CRMLSSold{start_month}_{end_month}.csv"

df.to_csv(OUTPUT_PATH, index=False)

print(f'Wrote {OUTPUT_PATH}')
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'\nColumns:')
for col in df.columns:
    print(f'  {col}')

Wrote ../../data/cleaned_CRMLSSold202512_202605.csv
Final shape: 61,604 rows x 77 columns

Columns:
  ListingKey
  CloseDate
  ClosePrice
  LivingArea
  BedroomsTotal
  BathroomsTotalInteger
  LotSizeAcres
  LotSizeAcres_imputed
  ViewYN
  BasementYN
  PoolPrivateYN
  AttachedGarageYN
  FireplaceYN
  NewConstructionYN
  CountyOrParish
  PostalCode
  ClosePrice_repaired
  County_Alameda
  County_Amador
  County_Butte
  County_Calaveras
  County_Colusa
  County_Contra_Costa
  County_El_Dorado
  County_Fresno
  County_Glenn
  County_Humboldt
  County_Imperial
  County_Inyo
  County_Kern
  County_Kings
  County_Lake
  County_Lassen
  County_Los_Angeles
  County_Madera
  County_Marin
  County_Mariposa
  County_Mendocino
  County_Merced
  County_Modoc
  County_Mono
  County_Monterey
  County_Napa
  County_Nevada
  County_Orange
  County_Other
  County_Placer
  County_Plumas
  County_Riverside
  County_Sacramento
  County_San_Benito
  County_San_Bernardino
  County_San_Diego
  County_San_Fran

- Cleaned CSV (`cleaned_CRMLSSold.csv`) is written into the gitignored `/data` folder, so the cleaned data also stays off the public repo
- The final dataset contains 61,604 rows and 77 columns, including repaired, encoded, and transformed features.

## Findings and Final Statements

This file will be applicable to any CRMLSSold*.csv file for cleaning.
Final .csv is NOT uploaded to the github.

### Summary
- Successfully developed a reusable preprocessing pipeline for CRMLSSold datasets.
- The pipeline performs filtering, data quality checks, missing value handling, feature selection, encoding, normalization, and chronological train/test splitting.
- Cleaning decisions are data-driven where possible (e.g., dynamic missing-value detection, Boolean detection, leakage prevention, and county encoding), allowing the notebook to generalize across future `CRMLSSold*.csv` files.

### Output
- Final cleaned dataset: 61,604 rows × 77 columns [for the (6) CRMLSSold*.csv of December 2025 to May 2026]
- Exported to: `../data`
- Dataset includes repaired, encoded, and transformed features while preserving audit columns for traceability in upcoming tasks

### Limitations
- This pipeline assumes the input follows the CRMLSSold schema.
- Thresholds (e.g., unit repair ratios, outlier bounds, training window size) are configurable and may require adjustment for future datasets or modeling objectives.
- Naming of the export file assumes the files used are consecutive months.